In [1]:
import pickle
import pandas as pd
import numpy as np
import sys
import os

sys.path.append(os.path.abspath(".."))
from sentence_transformers import SentenceTransformer, CrossEncoder

from utils import *

In [2]:
# Load artifacts that were saved during training

with open("../models/artifacts.pkl", "rb") as f:
    artifacts = pickle.load(f)

df = artifacts["df"]
embeddings = artifacts["embeddings"]
tfidf_vectorizer = artifacts["tfidf_vectorizer"]
tfidf_matrix = artifacts["tfidf_matrix"]

print("Artifacts loaded")
print(df.shape)

Artifacts loaded
(4009, 14)


In [3]:
# Load the bi-encoder and cross-encoder models

bi_encoder = SentenceTransformer("all-MiniLM-L6-v2")

cross_encoder = CrossEncoder("../models/cross_encoder_model")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

In [4]:
query = "Find me a lexus under 20k"
query = process_query(query)

candidates = recommend_cars(
    bi_encoder,
    embeddings,
    tfidf_vectorizer,
    tfidf_matrix,
    df,
    cross_encoder,
    query
)

print(candidates)

[{'description': 'Lexus IS 250 Base Petrol 204.0HP 2.5L V6 Cylinder Engine Gasoline Fuel Interior color: White exterior color: Black Automatic car 2010', 'brand': 'Lexus', 'model': 'IS 250 Base', 'price': 13250.0, 'engine': '204.0HP 2.5L V6 Cylinder Engine Gasoline Fuel', 'model_year': 2010, 'fuel_type': 'Petrol', 'transmission': 'Automatic', 'milage': 121250, 'int_col': 'White', 'ext_col': 'Black', 'score': -8.004948616027832}, {'description': 'Lexus IS 250 Base Petrol 204.0HP 2.5L V6 Cylinder Engine Gasoline Fuel Interior color: Black exterior color: Black Automatic car 2010', 'brand': 'Lexus', 'model': 'IS 250 Base', 'price': 10700.0, 'engine': '204.0HP 2.5L V6 Cylinder Engine Gasoline Fuel', 'model_year': 2010, 'fuel_type': 'Petrol', 'transmission': 'Automatic', 'milage': 132700, 'int_col': 'Black', 'ext_col': 'Black', 'score': -8.015788078308105}, {'description': 'Lexus IS 250 Base Petrol 204.0HP 2.5L V6 Cylinder Engine Gasoline Fuel Interior color: Black exterior color: Gray Auto

In [5]:

golden_test=[
    {
        'query': 'I want a Audi car under 30k 100000 km exterior color blue',
        'car': 'Audi SQ5 3.0T Premium Plus Petrol'
    },
    {
        'query': 'I want a Bmw car under 30k interior color black 2018 Automatic',
        'car': 'BMW 430 Gran Coupe i xDrive'
    },
    {
        'query': 'I want a Lexus model year 2021 engine 2.5',
        'car': 'Lexus ES 250 Base'
    },
      {
        'query': 'I want a Lexus model year 2021 engine 2.5 under 30k',
        'car': 'Lexus'
    },
    {
        'query': 'I want a Mercedes under 20k euros model year 2009 exterior color silver',
        'car': 'Mercedes-Benz CLK-Class CLK 350'
    },
    {
        'query': 'I want a Honda Civic under 10 000 Manual model year 2003',
        'car': 'Honda CR-V EX'
    },
      {
        'query': 'I want a Toyota from 2020 under 30k',
        'car': 'Toyota C-HR'
    },
    {
        'query': 'I want a Toyota from 2020 under 30k exterior color white',
        'car': 'Toyota 86 Base'
    },
    {
        'query': 'I want a Toyota from 2020 under 30k exterior color white motor 2.0',
        'car': 'Toyota C-HR LE'
    },
    {
        'query': 'I want a Mercedes under 20k 50000 kilometers 2009',
        'car': 'Mercedes-Benz CLK-Class'
    }
]


In [6]:

correct_keyword = 0
correct_semantic = 0
correct_hybrid = 0
total = len(golden_test)

for test in golden_test:
  query = test["query"]
  expected = test["car"]

  keyword_search_results = keyword_car_search(tfidf_vectorizer, tfidf_matrix, query,df)
  semantic_search_results = semantic_car_search(df, bi_encoder, query)
  hybrid_search_results = recommend_cars(
    bi_encoder,
    embeddings,
    tfidf_vectorizer,
    tfidf_matrix,
    df,
    cross_encoder,
    query
)

  if keyword_search_results is None or semantic_search_results is None or hybrid_search_results is None:
      continue

  keyword_top1 = keyword_search_results[0]["description"]
  semantic_top1 = semantic_search_results[0]["description"]
  hybrid_top1 = hybrid_search_results[0]["description"]

  if expected.lower() in keyword_top1.lower():
      correct_keyword += 1
  if expected.lower() in semantic_top1.lower():
      correct_semantic += 1
  if expected.lower() in hybrid_top1.lower():
      correct_hybrid += 1

accuracy_keyword = correct_keyword / total
accuracy_semantic = correct_semantic / total
accuracy_hybrid = correct_hybrid / total

print("Top-1 Accuracy for keyword search:", accuracy_keyword)
print("Top-1 Accuracy for semantic search:", accuracy_semantic)
print("Top-1 Accuracy for hybrid search:", accuracy_hybrid)

Top-1 Accuracy for keyword search: 0.4
Top-1 Accuracy for semantic search: 0.5
Top-1 Accuracy for hybrid search: 0.9
